# 03 — Models & Training Engine
Baselines: EfficientNet-B2, ResNet-50, ConvNeXt-Tiny, Swin-T. Proposed: backbone + one light attention (ECA/CBAM) at the stride-16 stage (24×24 @ 384) + original GAP + simple head. One shared training config.

In [1]:
import sys, json
from pathlib import Path
import torch
import pandas as pd

PROJECT_ROOT = Path("/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI")
SPLITS_DIR = PROJECT_ROOT / "splits"
CACHE_DIR  = PROJECT_ROOT / "cache_384"
SRC_DIR    = PROJECT_ROOT / "src"
RESULTS_DIR = PROJECT_ROOT / "results"
SRC_DIR.mkdir(exist_ok=True); RESULTS_DIR.mkdir(exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

Device: cuda | NVIDIA GeForce RTX 3050


In [2]:
MODELS_SRC = r'''
"""Baselines + proposed (backbone + one light attention at the stride-16 stage + GAP head)."""
import math
import torch
import torch.nn as nn
from torchvision import models as tvm

NUM_CLASSES = 3
DROPOUT = 0.3


class ECA(nn.Module):
    def __init__(self, ch, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(ch) + b) / gamma))
        k = t if t % 2 else t + 1
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)

    def forward(self, x):
        y = self.pool(x).squeeze(-1).transpose(-1, -2)               # B,1,C
        y = torch.sigmoid(self.conv(y)).transpose(-1, -2).unsqueeze(-1)  # B,C,1,1
        return x * y


class CBAM(nn.Module):
    def __init__(self, ch, r=16, k=7):
        super().__init__()
        mid = max(ch // r, 8)
        self.mlp = nn.Sequential(nn.Conv2d(ch, mid, 1, bias=False), nn.ReLU(inplace=True),
                                 nn.Conv2d(mid, ch, 1, bias=False))
        self.spatial = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)

    def forward(self, x):
        a = torch.sigmoid(self.mlp(x.mean((2, 3), keepdim=True)) + self.mlp(x.amax((2, 3), keepdim=True)))
        x = x * a
        s = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)
        return x * torch.sigmoid(self.spatial(s))


ATTN = {"eca": ECA, "cbam": CBAM}


def _head(in_f):
    return nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(in_f, NUM_CLASSES))


def _w(weights_enum, pretrained):
    return weights_enum.IMAGENET1K_V1 if pretrained else None


def _tv(name, pretrained):
    if name == "efficientnet_b2":
        return tvm.efficientnet_b2(weights=_w(tvm.EfficientNet_B2_Weights, pretrained))
    if name == "efficientnet_v2_s":
        return tvm.efficientnet_v2_s(weights=_w(tvm.EfficientNet_V2_S_Weights, pretrained))
    if name == "resnet50":
        return tvm.resnet50(weights=_w(tvm.ResNet50_Weights, pretrained))
    if name == "convnext_tiny":
        return tvm.convnext_tiny(weights=_w(tvm.ConvNeXt_Tiny_Weights, pretrained))
    if name == "swin_t":
        return tvm.swin_t(weights=_w(tvm.Swin_T_Weights, pretrained))
    raise ValueError(name)


def _set_head(m, name):
    if name == "efficientnet_b2":
        m.classifier = _head(1408)
    elif name == "efficientnet_v2_s":
        m.classifier = _head(1280)
    elif name == "resnet50":
        m.fc = _head(2048)
    elif name == "convnext_tiny":
        m.classifier[2] = _head(768)
    elif name == "swin_t":
        m.head = _head(768)
    return m


BASELINES = ["efficientnet_b2", "resnet50", "convnext_tiny", "swin_t"]
EXTRA_BASELINES = ["efficientnet_v2_s"]          # backbone-matched baseline for the proposed model
BACKBONES = {"convnext_tiny": 5, "efficientnet_v2_s": 5}   # index of the stride-16 stage in .features


def build_baseline(name, pretrained=True):
    return _set_head(_tv(name, pretrained), name)


@torch.no_grad()
def _stage_channels(features, idx):
    was = features.training
    features.eval()
    c = features[: idx + 1](torch.zeros(1, 3, 64, 64)).shape[1]
    features.train(was)
    return c


def build_proposed(backbone="convnext_tiny", attn="eca", pretrained=True):
    """Backbone + attention after the stride-16 stage (24x24 at 384px) + original norm/GAP + simple head."""
    m = _set_head(_tv(backbone, pretrained), backbone)
    idx = BACKBONES[backbone]
    ch = _stage_channels(m.features, idx)
    feats = list(m.features)
    m.features = nn.Sequential(*feats[: idx + 1], ATTN[attn](ch), *feats[idx + 1:])
    return m


def build_model(spec, pretrained=True):
    """spec: 'baseline:resnet50' or 'proposed:convnext_tiny:eca'"""
    parts = spec.split(":")
    if parts[0] == "baseline":
        return build_baseline(parts[1], pretrained)
    return build_proposed(parts[1], parts[2], pretrained)


def count_params(model):
    return sum(p.numel() for p in model.parameters())
'''
(SRC_DIR / "pneumo_models.py").write_text(MODELS_SRC, encoding="utf-8")
print("Saved:", SRC_DIR / "pneumo_models.py")

Saved: /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/src/pneumo_models.py


In [3]:
TRAIN_SRC = r'''
"""Shared training engine: AdamW + warmup/cosine + class-weighted CE (+LS) + AMP + early stopping."""
import json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt

DEFAULT_CFG = dict(epochs=50, lr=1e-4, weight_decay=1e-2, warmup_epochs=2, patience=10,
                   label_smoothing=0.1, amp=True, monitor="val_acc", grad_clip=1.0)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def make_scheduler(opt, epochs, warmup):
    warm = LinearLR(opt, start_factor=0.01, total_iters=warmup)
    cos = CosineAnnealingLR(opt, T_max=max(1, epochs - warmup))
    return SequentialLR(opt, [warm, cos], milestones=[warmup])


@torch.no_grad()
def predict(model, loader, device, amp=True):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
            out = model(x)
        ps.append(out.float().softmax(1).cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()


def metrics_from(y, p, class_weight=None, ls=0.0):
    pred = p.argmax(1)
    nll = -np.log(np.clip(p[np.arange(len(y)), y], 1e-8, 1)).mean()
    return dict(loss=float(nll), acc=accuracy_score(y, pred),
                bal_acc=balanced_accuracy_score(y, pred),
                macro_f1=f1_score(y, pred, average="macro"))


def train_run(model, train_loader, val_loader, class_weights, run_dir, device, cfg=None):
    cfg = {**DEFAULT_CFG, **(cfg or {})}
    run_dir = Path(run_dir); run_dir.mkdir(parents=True, exist_ok=True)
    model = model.to(device)
    crit = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=cfg["label_smoothing"])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = make_scheduler(opt, cfg["epochs"], cfg["warmup_epochs"])
    use_amp = cfg["amp"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    hist, best, best_ep, bad = [], -1.0, -1, 0
    t0 = time.time()
    for ep in range(cfg["epochs"]):
        model.train()
        tl, tc, tn = 0.0, 0, 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(x)
                loss = crit(out.float(), y)
            scaler.scale(loss).backward()
            if cfg["grad_clip"]:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(opt); scaler.update()
            tl += loss.item() * x.size(0); tn += x.size(0)
            if y.ndim == 1:
                tc += (out.argmax(1) == y).sum().item()
        sched.step()

        yv, pv = predict(model, val_loader, device, use_amp)
        m = metrics_from(yv, pv)
        row = dict(epoch=ep + 1, lr=opt.param_groups[0]["lr"], train_loss=tl / tn,
                   train_acc=(tc / tn) if tc else float("nan"),
                   val_loss=m["loss"], val_acc=m["acc"], val_bal_acc=m["bal_acc"], val_macro_f1=m["macro_f1"])
        hist.append(row)
        score = row[cfg["monitor"]]
        if score > best + 1e-6:
            best, best_ep, bad = score, ep + 1, 0
            torch.save(model.state_dict(), run_dir / "best.pt")
        else:
            bad += 1
        print(f"ep {ep+1:02d} | tr_loss {row['train_loss']:.4f} tr_acc {row['train_acc']:.4f} | "
              f"val_loss {row['val_loss']:.4f} val_acc {row['val_acc']:.4f} f1 {row['val_macro_f1']:.4f} "
              f"| best {best:.4f}@{best_ep} | {time.time()-t0:.0f}s")
        pd.DataFrame(hist).to_csv(run_dir / "history.csv", index=False)
        if bad >= cfg["patience"]:
            print("Early stopping."); break

    h = pd.DataFrame(hist)
    b = h.loc[h["epoch"] == best_ep].iloc[0]
    summary = dict(best_epoch=int(best_ep), best_val_acc=float(b["val_acc"]),
                   best_val_macro_f1=float(b["val_macro_f1"]), best_val_bal_acc=float(b["val_bal_acc"]),
                   epochs_run=int(len(h)), minutes=(time.time() - t0) / 60,
                   params=int(sum(p.numel() for p in model.parameters())), cfg=cfg)
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    plot_curves(h, run_dir / "curves.png")
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=device))
    return model, summary


def plot_curves(h, path):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h["epoch"], h["train_loss"], label="train"); ax[0].plot(h["epoch"], h["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(h["epoch"], h["train_acc"], label="train"); ax[1].plot(h["epoch"], h["val_acc"], label="val")
    ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.tight_layout(); plt.savefig(path, dpi=300, bbox_inches="tight"); plt.close(fig)
'''
(SRC_DIR / "pneumo_train.py").write_text(TRAIN_SRC, encoding="utf-8")
print("Saved:", SRC_DIR / "pneumo_train.py")

Saved: /mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI/src/pneumo_train.py


In [4]:

# ============================================================
# Smoke test: params, output shape, fwd+bwd time & VRAM (batch 16, 384px, AMP)
# ============================================================
import time, importlib
import pneumo_models, pneumo_train
importlib.reload(pneumo_models); importlib.reload(pneumo_train)
from pneumo_models import *

specs = [f"baseline:{n}" for n in BASELINES + EXTRA_BASELINES] + \
        [f"proposed:{b}:{a}" for b in BACKBONES for a in ATTN]

x = torch.randn(16, 3, 384, 384, device=device)
rows = []
for s in specs:
    m = build_model(s, pretrained=False).to(device).train()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t = time.time()
    with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
        out = m(x)
        loss = out.float().logsumexp(1).mean()
    loss.backward(); opt.step(); opt.zero_grad()
    if device.type == "cuda": torch.cuda.synchronize()
    rows.append({"model": s, "params_M": round(count_params(m) / 1e6, 2), "out": tuple(out.shape),
                 "step_s": round(time.time() - t, 2),
                 "vram_GB": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else None})
    del m, opt, out, loss
    if device.type == "cuda": torch.cuda.empty_cache()
pd.DataFrame(rows)


,model,params_M,out,step_s,vram_GB
0,baseline:efficientnet_b2,7.71,"(16, 3)",13.10,3.14
1,baseline:resnet50,23.51,"(16, 3)",0.35,2.27
2,baseline:convnext_tiny,27.82,"(16, 3)",10.15,3.15
3,baseline:swin_t,27.52,"(16, 3)",0.49,3.76
4,baseline:efficientnet_v2_s,20.18,"(16, 3)",4.68,3.82
5,proposed:convnext_tiny:eca,27.82,"(16, 3)",0.24,3.17
6,proposed:convnext_tiny:cbam,27.84,"(16, 3)",0.24,3.18
7,proposed:efficientnet_v2_s:eca,20.18,"(16, 3)",0.28,3.82
8,proposed:efficientnet_v2_s:cbam,20.18,"(16, 3)",0.28,3.83
